# From Transactions to Retention: Finding Revenue Leaks in Online Retail**Business question:** What can transaction history tell us about monthly sales, cancellations, and repeat customers?This short project uses real transaction records from the UCI Online Retail dataset. It demonstrates practical data cleaning, grouping, and summary statistics with pandas and NumPy.

In [ ]:
from pathlib import Pathimport numpy as npimport pandas as pd

## 1. Load the dataThe compressed CSV is included under `data/`. It contains invoice-line records rather than one row per order.

In [ ]:
data_path = Path('data') / 'online_retail.csv.gz'retail = pd.read_csv(data_path, parse_dates=['InvoiceDate'])retail.columns = retail.columns.str.strip()print(f'{retail.shape[0]:,} rows and {retail.shape[1]} columns')retail.head()

## 2. Prepare completed salesInvoices beginning with `C` represent cancellations. We keep those rows separate, then define completed sales as positive-quantity, positive-price lines. Customer IDs can be missing, so sales totals retain those rows; customer-level analysis later uses only identified customers.

In [ ]:
is_cancelled = retail['InvoiceNo'].astype(str).str.startswith('C')cancellations = retail.loc[is_cancelled].copy()sales = retail.loc[    ~is_cancelled & (retail['Quantity'] > 0) & (retail['UnitPrice'] > 0)].copy()sales['Revenue'] = sales['Quantity'] * sales['UnitPrice']print(f'Completed sales lines: {len(sales):,}')print(f'Rows missing CustomerID: {sales["CustomerID"].isna().sum():,}')sales[['InvoiceNo', 'InvoiceDate', 'Description', 'Quantity', 'UnitPrice', 'Revenue']].head()

## 3. Sales over timeNumPy gives a quick view of the distribution of line-item revenue. Monthly totals make broad demand changes easier to spot than individual transactions.

In [ ]:
line_revenue = sales['Revenue'].to_numpy()print(f'Median line revenue: ?{np.percentile(line_revenue, 50):,.2f}')print(f'95th percentile line revenue: ?{np.percentile(line_revenue, 95):,.2f}')monthly_revenue = (    sales.assign(Month=sales['InvoiceDate'].dt.to_period('M').dt.to_timestamp())    .groupby('Month')['Revenue']    .sum())monthly_revenue.tail(6).rename('Revenue (?)').to_frame()

## 4. Cancellation signalsWe estimate cancelled value from cancellation lines with negative quantities and positive prices. The rate below compares cancelled invoices with all observed completed and cancelled invoices. It is an invoice-level indicator, not proof of why an order was cancelled.

In [ ]:
cancel_lines = cancellations.loc[    (cancellations['Quantity'] < 0) & (cancellations['UnitPrice'] > 0)]completed_invoice_count = sales['InvoiceNo'].nunique()cancelled_invoice_count = cancellations['InvoiceNo'].nunique()observed_invoice_count = completed_invoice_count + cancelled_invoice_countcancellation_rate = cancelled_invoice_count / observed_invoice_countestimated_cancelled_value = (-cancel_lines['Quantity'] * cancel_lines['UnitPrice']).sum()print(f'Completed invoices: {completed_invoice_count:,}')print(f'Cancelled invoices: {cancelled_invoice_count:,}')print(f'Cancellation invoice rate: {cancellation_rate:.1%}')print(f'Estimated cancelled value: ?{estimated_cancelled_value:,.2f}')

## 5. Markets and customer retentionThe international ranking excludes the United Kingdom to make other markets visible. A repeat customer is defined simply as a known customer with at least two distinct completed invoices.

In [ ]:
international_revenue = (    sales.loc[sales['Country'] != 'United Kingdom']    .groupby('Country')['Revenue']    .sum()    .sort_values(ascending=False)    .head(10))display(international_revenue.rename('Revenue (?)').to_frame())known_customer_sales = sales.dropna(subset=['CustomerID']).copy()invoices_per_customer = known_customer_sales.groupby('CustomerID')['InvoiceNo'].nunique()repeat_customer_ids = invoices_per_customer.index[invoices_per_customer >= 2]known_customer_sales['CustomerType'] = np.where(    known_customer_sales['CustomerID'].isin(repeat_customer_ids), 'Repeat', 'One-time')repeat_share = len(repeat_customer_ids) / len(invoices_per_customer)print(f'Repeat customers: {len(repeat_customer_ids):,} of {len(invoices_per_customer):,} identified customers ({repeat_share:.1%})')customer_summary = known_customer_sales.groupby('CustomerType').agg(    Customers=('CustomerID', 'nunique'), Revenue=('Revenue', 'sum'))customer_summary['RevenueShare'] = customer_summary['Revenue'] / customer_summary['Revenue'].sum()customer_summary

## 6. Findings to take into a business discussionUse the outputs above to write three evidence-based takeaways: one about the monthly pattern, one about cancellations, and one about repeat customers or international markets. Include the metric that supports each takeaway.**Limitations:** This is descriptive analysis, so patterns do not establish causes. Cancelled value is estimated from cancellation lines, invoice rate is a simple invoice-count proxy, and customer analysis excludes transactions without a CustomerID.